In [11]:
from pyspark.sql import SparkSession
import json
import os
from pyspark.sql import functions as sf
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "forge-commerce-user")
SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "forge-commerce-pass")
S3_ENDPOINT = os.environ.get("AWS_S3_ENDPOINT", "http://minio:9000")
ORDERS = "orders"
ORDER_ITEMS = "order_items"
PRODUCTS = "products"
CUSTOMERS = "customers"
RAW_BUCKET = "raw"
CLEANED_BUCKET = "cleaned"
CURATED_BUCKET = "curated"
RAW_PATH = f"s3a://{RAW_BUCKET}/"
CLEANED_PATH = f"s3a://{CLEANED_BUCKET}/"
CURATED_PATH = f"s3a://{CURATED_BUCKET}/"

In [12]:
spark = (
        SparkSession.builder.appName("test_order_items")
        .master(os.environ.get("SPARK_MASTER", "spark://spark-master:7077"))
        .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
        .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
        .config("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        # Delta Lake configurations
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
        .getOrCreate()
    )

In [13]:
order_items = spark.read.format("delta").load(CURATED_PATH + ORDER_ITEMS)
order_items.printSchema()

# get product id with most sales
product_id = order_items.groupBy("product_id").count().orderBy("count", ascending=False).first()[0]
print(product_id)
filter_expression = sf.col("product_id") == product_id


root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sk_product: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_percentage: decimal(10,2) (nullable = true)
 |-- line_total: decimal(10,2) (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)



207


In [15]:
products = spark.read.format("delta").load(CURATED_PATH + PRODUCTS)

products.filter(filter_expression).orderBy("product_id", sf.col("created_at").asc()).select("product_id", "sk_product", "effective_from", "effective_to", "is_active").show(10)

order_items.filter(filter_expression).orderBy("order_id", sf.col("created_at").asc()).select("order_id", "product_id", "sk_product", "created_at").show(20)

+----------+----------+-------------------+-------------------+---------+
|product_id|sk_product|     effective_from|       effective_to|is_active|
+----------+----------+-------------------+-------------------+---------+
|       207|       413|2026-03-28 21:22:13|2026-03-28 21:22:22|    false|
|       207|       414|2026-03-28 21:22:22|2026-03-28 21:30:15|    false|
|       207|      2620|2026-03-28 21:30:15|2026-03-28 21:30:24|    false|
|       207|      2621|2026-03-28 21:30:24|               NULL|     true|
+----------+----------+-------------------+-------------------+---------+



+--------+----------+----------+-------------------+
|order_id|product_id|sk_product|         created_at|
+--------+----------+----------+-------------------+
|     210|       207|       413|2026-03-28 21:22:15|
|     271|       207|       413|2026-03-28 21:22:15|
|     275|       207|       413|2026-03-28 21:22:15|
|     291|       207|       413|2026-03-28 21:22:15|
|     315|       207|       413|2026-03-28 21:22:15|
|     376|       207|       413|2026-03-28 21:22:15|
|     739|       207|       413|2026-03-28 21:22:15|
|     823|       207|       413|2026-03-28 21:22:15|
|     961|       207|       413|2026-03-28 21:22:15|
|    1041|       207|       414|2026-03-28 21:22:23|
|    1207|       207|       414|2026-03-28 21:22:23|
|    1271|       207|       414|2026-03-28 21:22:23|
|    1275|       207|       414|2026-03-28 21:22:23|
|    1291|       207|       414|2026-03-28 21:22:23|
|    1318|       207|       414|2026-03-28 21:22:23|
|    1375|       207|       414|2026-03-28 21:

In [16]:
products.orderBy(sf.col("created_at").desc()).select("product_id", "sk_product", "created_at", "effective_from", "effective_to", "is_active").show(10, truncate=False)

+----------+----------+-------------------+-------------------+------------+---------+
|product_id|sk_product|created_at         |effective_from     |effective_to|is_active|
+----------+----------+-------------------+-------------------+------------+---------+
|34        |2102      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|64        |2192      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|38        |2114      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|3         |2009      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|39        |2117      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|13        |2039      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|41        |2123      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|19        |2057      |2026-03-28 21:30:24|2026-03-28 21:30:24|NULL        |true     |
|44        |2132      |2026-03-28 21:30:24|

In [18]:
product_raw = spark.read.format("json").load(RAW_PATH + PRODUCTS)
product_raw.orderBy(sf.col("created_at").desc()).select("product_id", "created_at").show(10, truncate=False)

+----------+-------------------+
|product_id|created_at         |
+----------+-------------------+
|1         |2026-03-28 21:30:24|
|2         |2026-03-28 21:30:24|
|3         |2026-03-28 21:30:24|
|4         |2026-03-28 21:30:24|
|5         |2026-03-28 21:30:24|
|6         |2026-03-28 21:30:24|
|7         |2026-03-28 21:30:24|
|8         |2026-03-28 21:30:24|
|9         |2026-03-28 21:30:24|
|10        |2026-03-28 21:30:24|
+----------+-------------------+
only showing top 10 rows



In [19]:
spark.stop()